In [1]:
import torch
from torch.utils.data import DataLoader

In [2]:
# need to add path using os and sys first
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

In [3]:
from utils.tokenizer import prepare_tokenizer, pad_tensor
from utils.load_data import load_data, prep_dolly_collate_fn, prep_squad_collate_fn

In [4]:
vocab, eos_idx, bos_idx, vocab_size = prepare_tokenizer()

In [5]:
qc_len = 512 + 256
a_len = 100

In [6]:
squad_collate = prep_squad_collate_fn(qc_len, a_len, eos_idx, bos_idx, pad_tensor)

In [7]:
squad, dolly = load_data()

In [8]:
batch_size = 2

In [9]:
squadDataloader = DataLoader(squad, batch_size=batch_size, shuffle=True, collate_fn=squad_collate)

In [10]:
embedding_dim = 128
num_heads = 4
phm_factor = 4
lm_head_factor = 2
num_encoder_layers = 4
num_decoder_layers = 3

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
from models.model import VanillaEncoderDecoder

In [13]:
# embedding_dim, num_heads, num_encoder_layers, num_decoder_layers, vocab_size, factor, lm_head_factor, eos_idx, bos_idx):
model = VanillaEncoderDecoder(embedding_dim, num_heads, num_encoder_layers, num_decoder_layers, vocab_size, phm_factor, lm_head_factor, eos_idx, bos_idx)
model = model.to(device)

In [14]:
for batch in squadDataloader:
    break

In [15]:
questioncontext, answer = batch
questioncontext = questioncontext.to(device)
answer = answer.to(device)
a_input, a_target = answer[:, :-1], answer[:, 1:]

In [16]:
prediction = model(questioncontext, a_input)

In [17]:
loss = torch.nn.functional.cross_entropy(prediction.reshape(-1, vocab_size), a_target.reshape(-1))

In [18]:
# ensure we can use memory efficient attention
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    prediction = model(questioncontext, a_input)

In [19]:
loss = torch.nn.functional.cross_entropy(prediction.reshape(-1, vocab_size), a_target.reshape(-1))